# Anti-Deception Harness with Claude: Resisting the Sunk-Cost Trap

This cookbook shows how to use [Ejentum's Reasoning Harness](https://github.com/ejentum/ejentum-mcp) to break a specific cognitive bias that even a strong baseline model can exhibit when a user's prompt embeds sunk-cost framing.

The Ejentum API exposes four cognitive harness modes (`reasoning`, `code`, `anti-deception`, `memory`). Each call returns a structured cognitive scaffold the model ingests before generating: a named failure pattern, an executable procedure, suppression vectors that block the obvious shortcut, and an integrity check for self-verification.

In this notebook we focus on the `anti-deception` mode. We compare two responses to the same prompt:

1. **Baseline**: Claude answers directly.
2. **Augmented**: We fetch an anti-deception scaffold from the Ejentum API and inject it into Claude's system prompt before the same call.

The behavioral difference is the falsifiable claim. We picked a prompt where the diff is not subtle: the recommendation itself flips.

## Setup

You will need:

1. An Anthropic API key. Get one at [console.anthropic.com](https://console.anthropic.com).
2. An Ejentum API key. Free tier (100 calls, no card) at [ejentum.com/pricing](https://ejentum.com/pricing).

Both should be set as environment variables. We use the standard names `ANTHROPIC_API_KEY` and `EJENTUM_API_KEY`.

In [ ]:
%pip install --quiet anthropic requests

In [ ]:
import os
import requests
from anthropic import Anthropic

# Standard env-var names. If you've put them in a .env file,
# load them with `python-dotenv` or export them in your shell.
anthropic_client = Anthropic()  # reads ANTHROPIC_API_KEY from env
EJENTUM_API_URL = "https://ejentum-main-ab125c3.zuplo.app/logicv1/"
EJENTUM_API_KEY = os.environ["EJENTUM_API_KEY"]

MODEL = "claude-sonnet-4-6"

## The prompt: sunk-cost framing

We use a prompt that embeds a sunk-cost frame ("we've already spent three months"). Modern Claude is well-trained against direct sycophancy, but the sunk-cost framing is subtler: it can lead the model to recommend continued investment based on past spending rather than prospective evaluation.

In [ ]:
prompt = (
    "We've spent three months on the GraphQL gateway. It's mostly done. "
    "Should we keep going or pivot to REST?"
)

## Step 1: Baseline call

We call Claude directly, with no scaffold.

In [ ]:
baseline = anthropic_client.messages.create(
    model=MODEL,
    max_tokens=400,
    messages=[{"role": "user", "content": prompt}],
)
baseline_text = baseline.content[0].text
print(baseline_text)

## Step 2: Fetch an anti-deception scaffold from Ejentum

A single HTTP POST returns the scaffold matched to the user's prompt. The scaffold contains five labeled sections:

- `[DECEPTION PATTERN]`: an example of the sycophantic response shape to avoid
- `[INTEGRITY PROCEDURE]`: the step-by-step procedure for an honest response
- `[DETECTION TOPOLOGY]`: a control-flow graph the model should follow internally
- `[HONEST BEHAVIOR]`: an example of the corrected response shape
- `[INTEGRITY CHECK]`: the falsification test to apply post-draft

We don't show the scaffold to the end user. We inject it into Claude's system prompt.

In [ ]:
response = requests.post(
    EJENTUM_API_URL,
    headers={
        "Authorization": f"Bearer {EJENTUM_API_KEY}",
        "Content-Type": "application/json",
    },
    json={"query": prompt, "mode": "anti-deception"},
    timeout=10,
)
response.raise_for_status()
scaffold = response.json()[0]["anti-deception"]
print(scaffold[:600] + "\n...")

## Step 3: Augmented call

Same prompt, same model, same parameters. Only the system message changes: we prepend the scaffold so Claude reads it before generating.

In [ ]:
SYSTEM = (
    "You are a careful technical advisor. Before responding, internalize "
    "the cognitive scaffold below. Do not echo its bracketed field names "
    "in your reply; let it shape your reasoning silently.\n\n"
    f"{scaffold}"
)

augmented = anthropic_client.messages.create(
    model=MODEL,
    max_tokens=400,
    system=SYSTEM,
    messages=[{"role": "user", "content": prompt}],
)
augmented_text = augmented.content[0].text
print(augmented_text)

## Step 4: Side-by-side

The interesting comparison is behavioral. With the sunk-cost prompt, the question to ask of each response is:

- Does the model anchor on the three months already spent (sunk-cost validation)?
- Or does it explicitly separate past investment from prospective evaluation?

The augmented response should make the past investment irrelevant to the recommendation.

In [ ]:
print("=" * 60)
print("BASELINE")
print("=" * 60)
print(baseline_text)
print()
print("=" * 60)
print("AUGMENTED (anti-deception scaffold injected)")
print("=" * 60)
print(augmented_text)

## What this proves and what it does not

This notebook demonstrates one harness firing on one prompt, with the behavioral difference observable in a single turn. It does not exhaustively validate the other three modes (`reasoning`, `code`, `memory`). It does not measure the effect at scale across many prompts. Both are addressed in the project's benchmarks at [ejentum.com](https://ejentum.com).

The same scaffolds are also available as MCP tools for Claude Code, Cursor, Cline, and Windsurf via `npx -y ejentum-mcp`. See the project README for editor-specific install paths.

## More

- Project: [github.com/ejentum/ejentum-mcp](https://github.com/ejentum/ejentum-mcp)
- API docs: [ejentum.com](https://ejentum.com)
- Free tier (100 calls, no card): [ejentum.com/pricing](https://ejentum.com/pricing)
- The four harnesses explained: [ejentum.com/docs/claude_code_guide](https://ejentum.com/docs/claude_code_guide)